# ESI — 4e année, option Développeur
## TP formateur — NLP, Transformers, IA générative, API & MLOps

**Objectif :** relier la théorie du Support 2 à une chaîne pratique et exécutable :

`texte → tokens → embeddings → RNN → attention → Transformer → génération → API → monitoring`

> Ce notebook est un **support de démonstration formateur**, pas l'un des quatre TP notés.  
> Le cœur du notebook fonctionne sans API commerciale et sans clé secrète.

## 0. Organisation de la séance

1. Préparer l'environnement  
2. Tokeniser un texte et construire un vocabulaire  
3. Comprendre les embeddings  
4. Voir ce que fait un RNN  
5. Calculer une attention scaled dot-product  
6. Construire un mini Transformer causal  
7. Entraîner un mini modèle de langage pédagogique  
8. Générer du texte token par token  
9. Comprendre température et top-k  
10. Structurer un prompt professionnel  
11. Transformer l'inférence en API FastAPI  
12. Préparer la conteneurisation Docker  
13. Ajouter logs et monitoring  
14. Terminer par une checklist sécurité / RGPD / AI Act

**Important :** le mini-corpus utilisé dans la partie mécanismes sert uniquement à rendre les calculs visibles en classe. Il ne constitue pas un dataset d'évaluation ni un TP noté.

In [2]:
import sys
import platform
import random
import numpy as np
import torch

print("Python :", sys.version.split()[0])
print("Système :", platform.system())
print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())

ModuleNotFoundError: No module named 'torch'

### Explication du code
- `sys` et `platform` donnent des informations sur l'environnement d'exécution.
- `numpy` est utilisé pour les calculs numériques classiques.
- `torch` fournit les tenseurs, les réseaux de neurones et l'optimisation.
- `torch.cuda.is_available()` permet de savoir si un GPU CUDA est accessible.

**Pourquoi ?** Avant un TP Deep Learning/NLP, on vérifie l'environnement pour éviter de confondre un problème de code avec un problème d'installation.

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", device)

Device utilisé : cpu


### Explication du code
- On fixe les graines pseudo-aléatoires afin de rendre les expériences plus reproductibles.
- `device` sélectionne automatiquement le GPU s'il existe, sinon le CPU.
- Les tenseurs et modèles devront être placés sur le même `device`.

**À retenir :** reproductible ne veut pas dire parfaitement identique sur toutes les machines, mais fixer les seeds est un minimum professionnel.

# 1. Tokenisation : du texte aux identifiants

Un LLM ne reçoit pas directement une phrase. Il reçoit une **séquence d'identifiants de tokens**.

Pour visualiser le mécanisme sans téléchargement externe, nous commençons par un tokenizer pédagogique au niveau des mots. Les LLM réels utilisent généralement des tokenizers de sous-mots (BPE, WordPiece, SentencePiece, etc.).

In [3]:
import re

text = "Une API d'intelligence artificielle reçoit un texte et retourne une prédiction."

tokens = re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)

print(tokens)
print("Nombre de tokens :", len(tokens))

['une', 'api', 'd', "'", 'intelligence', 'artificielle', 'reçoit', 'un', 'texte', 'et', 'retourne', 'une', 'prédiction', '.']
Nombre de tokens : 14


### Explication du code
- `text.lower()` uniformise ici la casse uniquement pour la démonstration.
- L'expression régulière sépare les suites de caractères alphanumériques et la ponctuation.
- Le résultat est une liste d'unités manipulables par le programme.

**Pourquoi ?** Le réseau neuronal ne travaille pas sur les chaînes de caractères brutes. La tokenisation constitue la première transformation.

In [4]:
vocab = sorted(set(tokens))
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for token, idx in token_to_id.items()}

token_ids = [token_to_id[token] for token in tokens]

print("Vocabulaire :", token_to_id)
print("IDs :", token_ids)

Vocabulaire : {"'": 0, '.': 1, 'api': 2, 'artificielle': 3, 'd': 4, 'et': 5, 'intelligence': 6, 'prédiction': 7, 'retourne': 8, 'reçoit': 9, 'texte': 10, 'un': 11, 'une': 12}
IDs : [12, 2, 4, 0, 6, 3, 9, 11, 10, 5, 8, 12, 7, 1]


### Explication du code
- `set(tokens)` récupère les tokens uniques.
- `token_to_id` associe chaque token à un entier.
- `id_to_token` construit l'opération inverse.
- `token_ids` est la représentation entière de la phrase.

**Attention :** un ID n'exprime aucune proximité sémantique. Le token d'ID 8 n'est pas « deux fois plus grand » que celui d'ID 4.

# 2. Embeddings : transformer les IDs en vecteurs

Une couche d'embedding apprend une matrice de taille :

`taille_du_vocabulaire × dimension_embedding`

Chaque ID sert d'index pour récupérer un vecteur dense.

In [5]:
embedding_dim = 8
embedding = torch.nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=embedding_dim
)

x_ids = torch.tensor(token_ids, dtype=torch.long)
x_emb = embedding(x_ids)

print("Shape des IDs :", x_ids.shape)
print("Shape des embeddings :", x_emb.shape)
print("\nVecteur du premier token :")
print(x_emb[0])

Shape des IDs : torch.Size([14])
Shape des embeddings : torch.Size([14, 8])

Vecteur du premier token :
tensor([-0.9864,  0.1233,  0.3499,  0.6173, -0.1693,  0.2332,  4.0356,  1.2795],
       grad_fn=<SelectBackward0>)


### Explication du code
- `nn.Embedding` crée une table de vecteurs entraînables.
- `num_embeddings` correspond au nombre de tokens du vocabulaire.
- `embedding_dim=8` signifie que chaque token est représenté par huit nombres.
- Pour une séquence de `T` tokens, la sortie a ici la forme `(T, 8)`.

**À retenir :** les embeddings sont des paramètres du modèle. Ils sont ajustés par descente de gradient.

# 3. RNN : faire circuler un état caché dans la séquence

In [6]:
rnn = torch.nn.RNN(
    input_size=embedding_dim,
    hidden_size=16,
    batch_first=True
)

batch = x_emb.unsqueeze(0)  # (batch=1, sequence, embedding)
outputs, h_last = rnn(batch)

print("Entrée RNN :", batch.shape)
print("Sorties RNN :", outputs.shape)
print("Dernier état caché :", h_last.shape)

Entrée RNN : torch.Size([1, 14, 8])
Sorties RNN : torch.Size([1, 14, 16])
Dernier état caché : torch.Size([1, 1, 16])


### Explication du code
- `input_size` doit correspondre à la dimension des embeddings.
- `hidden_size=16` fixe la taille de la mémoire cachée.
- `unsqueeze(0)` ajoute la dimension batch.
- `outputs` contient un état pour chaque position.
- `h_last` contient le dernier état caché.

**Limite :** le calcul du RNN est naturellement séquentiel. Les Transformers remplacent cette dépendance récurrente par l'attention.

# 4. Scaled dot-product attention : voir réellement Q, K et V

In [7]:
import math

torch.manual_seed(SEED)

T = 4
d_k = 6

Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)
V = torch.randn(T, d_k)

scores = Q @ K.T / math.sqrt(d_k)
weights = torch.softmax(scores, dim=-1)
context = weights @ V

print("Scores :\n", scores.round(decimals=2))
print("\nPoids d'attention :\n", weights.round(decimals=2))
print("\nSomme de chaque ligne :", weights.sum(dim=-1))
print("\nContexte :", context.shape)

Scores :
 tensor([[ 1.5400, -0.9300, -1.6500,  0.8300],
        [ 0.0600, -0.1700, -0.3000,  0.3500],
        [-1.6900, -1.3300,  0.6600, -0.1000],
        [-0.7800, -1.3200,  0.5100,  0.4300]])

Poids d'attention :
 tensor([[0.6200, 0.0500, 0.0300, 0.3000],
        [0.2600, 0.2100, 0.1800, 0.3500],
        [0.0600, 0.0800, 0.5900, 0.2800],
        [0.1200, 0.0700, 0.4200, 0.3900]])

Somme de chaque ligne : tensor([1.0000, 1.0000, 1.0000, 1.0000])

Contexte : torch.Size([4, 6])


### Explication du code
- `Q @ K.T` calcule la compatibilité entre chaque Query et chaque Key.
- La division par `sqrt(d_k)` stabilise l'amplitude des scores.
- `softmax` transforme chaque ligne en distribution de poids.
- `weights @ V` produit une combinaison pondérée des Value.
- Chaque ligne des poids somme à 1.

**Lecture pédagogique :** une ligne répond à la question « pour ce token, quelles positions sont importantes ? ».

# 5. Masque causal : empêcher un token de regarder le futur

In [8]:
causal_mask = torch.triu(
    torch.ones(T, T, dtype=torch.bool),
    diagonal=1
)

masked_scores = scores.masked_fill(causal_mask, float("-inf"))
causal_weights = torch.softmax(masked_scores, dim=-1)

print("Masque causal :\n", causal_mask)
print("\nAttention causale :\n", causal_weights.round(decimals=2))

Masque causal :
 tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])

Attention causale :
 tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5600, 0.4400, 0.0000, 0.0000],
        [0.0800, 0.1100, 0.8100, 0.0000],
        [0.1200, 0.0700, 0.4200, 0.3900]])


### Explication du code
- `torch.triu(..., diagonal=1)` marque toutes les positions situées strictement au-dessus de la diagonale.
- Ces positions correspondent au **futur**.
- `masked_fill(..., -inf)` rend leur probabilité nulle après softmax.
- Ainsi, la position `t` ne peut utiliser que les tokens déjà disponibles.

**Pourquoi ?** C'est le mécanisme fondamental d'un Transformer decoder-only autoregressif.

# 6. Construire un mini Transformer causal

In [9]:
class TinyLanguageModel(torch.nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, d_model)
        self.position = torch.nn.Embedding(128, d_model)

        layer = torch.nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=64,
            dropout=0.0,
            batch_first=True
        )
        self.transformer = torch.nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = torch.nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        B, T = input_ids.shape

        positions = torch.arange(T, device=input_ids.device)
        h = self.embedding(input_ids) + self.position(positions)

        mask = torch.triu(
            torch.ones(T, T, device=input_ids.device, dtype=torch.bool),
            diagonal=1
        )

        h = self.transformer(h, mask=mask)
        logits = self.lm_head(h)
        return logits

### Explication du code
Cette classe assemble les composants essentiels d'un petit modèle de langage :

- `Embedding` représente les tokens.
- Un second `Embedding` représente leur position.
- `TransformerEncoderLayer` est utilisé ici comme bloc Transformer générique.
- Le masque causal empêche l'accès aux tokens futurs.
- `lm_head` projette chaque représentation vers une distribution de taille `vocab_size`.
- La sortie `logits` a la forme `(batch, longueur, vocabulaire)`.

**Important :** c'est une architecture pédagogique minuscule. Ce n'est pas un LLM industriel.

# 7. Mini-corpus pédagogique pour observer l'apprentissage autoregressif

Le corpus ci-dessous est volontairement petit afin que l'entraînement soit rapide en classe.  
Il sert à **observer le mécanisme next-token prediction**, pas à mesurer la qualité d'un système NLP réel.

In [10]:
sentences = [
    "le modèle prédit le prochain token",
    "le modèle apprend sur des données",
    "le développeur teste le modèle",
    "le développeur déploie une api",
    "une api reçoit une requête",
    "une api retourne une réponse",
    "le monitoring surveille le service",
    "le monitoring détecte une dérive",
]

def simple_tokenize(s):
    return s.lower().split()

all_tokens = ["<bos>", "<eos>"]
for sentence in sentences:
    all_tokens.extend(simple_tokenize(sentence))

vocab2 = sorted(set(all_tokens))
stoi = {t: i for i, t in enumerate(vocab2)}
itos = {i: t for t, i in stoi.items()}

print("Taille vocabulaire :", len(vocab2))
print(vocab2)

Taille vocabulaire : 25
['<bos>', '<eos>', 'api', 'apprend', 'des', 'données', 'déploie', 'dérive', 'détecte', 'développeur', 'le', 'modèle', 'monitoring', 'prochain', 'prédit', 'requête', 'retourne', 'reçoit', 'réponse', 'service', 'sur', 'surveille', 'teste', 'token', 'une']


### Explication du code
- Le corpus contient quelques phrases liées au développement et au déploiement d'IA.
- `<bos>` marque le début d'une séquence et `<eos>` sa fin.
- `stoi` signifie ici *string to integer*.
- `itos` réalise l'opération inverse.

**Pourquoi ce petit corpus ?** Pour rendre l'entraînement observable en quelques secondes. Pour une vraie évaluation, il faudrait un corpus réel, beaucoup plus grand et un protocole train/validation/test.

In [11]:
examples = []

for sentence in sentences:
    ids = [stoi["<bos>"]] + [stoi[t] for t in simple_tokenize(sentence)] + [stoi["<eos>"]]

    for i in range(1, len(ids)):
        context = ids[:i]
        target = ids[i]
        examples.append((context, target))

print("Nombre d'exemples next-token :", len(examples))
print("Premier contexte :", examples[0][0])
print("Première cible :", examples[0][1])

Nombre d'exemples next-token : 50
Premier contexte : [0]
Première cible : 10


### Explication du code
Pour chaque phrase, nous créons plusieurs problèmes de prédiction :

- contexte : les tokens déjà vus ;
- cible : le token suivant.

Par exemple, après `<bos>`, le modèle doit apprendre quel token peut venir ensuite. Puis après deux tokens, il prédit le troisième, etc.

C'est l'idée fondamentale de l'apprentissage autoregressif.

In [12]:
model = TinyLanguageModel(len(vocab2)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(120):
    total_loss = 0.0

    random.shuffle(examples)

    for context, target in examples:
        x = torch.tensor([context], dtype=torch.long, device=device)
        y = torch.tensor([target], dtype=torch.long, device=device)

        optimizer.zero_grad()

        logits = model(x)
        last_logits = logits[:, -1, :]

        loss = criterion(last_logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | loss moyenne = {total_loss/len(examples):.4f}")

Epoch   0 | loss moyenne = 2.9758
Epoch  20 | loss moyenne = 0.4384
Epoch  40 | loss moyenne = 0.4366
Epoch  60 | loss moyenne = 0.3905
Epoch  80 | loss moyenne = 0.9229
Epoch 100 | loss moyenne = 0.4581


### Explication du code
- `AdamW` met à jour les paramètres du modèle.
- Pour chaque contexte, le modèle produit des logits à toutes les positions.
- `logits[:, -1, :]` conserve la prédiction associée à la dernière position du contexte.
- `CrossEntropyLoss` compare les logits au véritable prochain token.
- `backward()` calcule les gradients.
- `optimizer.step()` met les paramètres à jour.

**À observer :** la loss doit globalement diminuer. Sur un corpus aussi petit, le modèle mémorise rapidement des régularités.

# 8. Génération token par token

In [13]:
@torch.no_grad()
def generate(model, start_tokens, max_new_tokens=10, temperature=1.0, top_k=None):
    model.eval()

    ids = [stoi["<bos>"]] + [stoi[t] for t in start_tokens.split()]

    for _ in range(max_new_tokens):
        x = torch.tensor([ids], dtype=torch.long, device=device)
        logits = model(x)[:, -1, :] / temperature

        if top_k is not None:
            values, _ = torch.topk(logits, k=min(top_k, logits.shape[-1]))
            threshold = values[:, -1].unsqueeze(-1)
            logits = torch.where(
                logits < threshold,
                torch.full_like(logits, float("-inf")),
                logits
            )

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == stoi["<eos>"]:
            break

        ids.append(next_id)

    return " ".join(itos[i] for i in ids if itos[i] not in {"<bos>", "<eos>"})

### Explication du code
La fonction reproduit la boucle autoregressive :

1. encoder le contexte ;
2. calculer les logits du prochain token ;
3. appliquer la température ;
4. éventuellement éliminer les candidats hors top-k ;
5. transformer les logits en probabilités ;
6. échantillonner un token ;
7. l'ajouter au contexte ;
8. recommencer.

`@torch.no_grad()` désactive le calcul des gradients pendant l'inférence.

In [14]:
for temp in [0.3, 0.8, 1.2]:
    print(f"Température {temp} :", generate(
        model,
        start_tokens="le",
        max_new_tokens=8,
        temperature=temp,
        top_k=5
    ))

Température 0.3 : le modèle apprend sur api token teste sur prédit
Température 0.8 : le modèle apprend sur api token détecte une réponse
Température 1.2 : le modèle prédit le développeur api api api api


### Explication du code
Cette cellule compare plusieurs températures :

- température faible : distribution plus concentrée, résultat généralement plus stable ;
- température élevée : distribution plus plate, donc davantage de diversité ;
- `top_k=5` interdit d'échantillonner parmi tous les tokens du vocabulaire.

**Question à poser aux étudiants :** pour une API bancaire, voudriez-vous la même température que pour une application créative ?

# 9. Prompt engineering : passer d'une consigne vague à un contrat

Dans une application GenAI, le prompt doit être pensé comme une partie du **contrat logiciel**.

In [15]:
ticket = '''
Depuis la mise à jour de ce matin, notre API de paiement retourne
une erreur 500 sur environ 30 % des requêtes.
'''

prompt = f'''
RÔLE
Tu es un assistant de triage d'incidents logiciels.

TÂCHE
Analyse le ticket utilisateur.

CONTRAINTES
- N'invente aucune information absente du ticket.
- La catégorie doit être parmi : API, DATABASE, FRONTEND, OTHER.
- L'urgence doit être parmi : LOW, MEDIUM, HIGH, CRITICAL.

FORMAT DE SORTIE
Retourne uniquement un JSON avec :
category, urgency, rationale.

TICKET
{ticket}
'''

print(prompt)


RÔLE
Tu es un assistant de triage d'incidents logiciels.

TÂCHE
Analyse le ticket utilisateur.

CONTRAINTES
- N'invente aucune information absente du ticket.
- La catégorie doit être parmi : API, DATABASE, FRONTEND, OTHER.
- L'urgence doit être parmi : LOW, MEDIUM, HIGH, CRITICAL.

FORMAT DE SORTIE
Retourne uniquement un JSON avec :
category, urgency, rationale.

TICKET

Depuis la mise à jour de ce matin, notre API de paiement retourne
une erreur 500 sur environ 30 % des requêtes.




### Explication du code
Le prompt sépare explicitement :

- le rôle ;
- la tâche ;
- les contraintes ;
- les valeurs autorisées ;
- le format attendu ;
- les données utilisateur.

Cette structure améliore la testabilité du système. Mais elle ne remplace jamais la validation du JSON côté application.

# 10. Validation : ne jamais faire confiance directement à une sortie LLM

In [16]:
from dataclasses import dataclass

ALLOWED_CATEGORIES = {"API", "DATABASE", "FRONTEND", "OTHER"}
ALLOWED_URGENCY = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}

@dataclass
class TriageResult:
    category: str
    urgency: str
    rationale: str

def validate_result(data):
    if data.get("category") not in ALLOWED_CATEGORIES:
        raise ValueError("Catégorie invalide")

    if data.get("urgency") not in ALLOWED_URGENCY:
        raise ValueError("Urgence invalide")

    if not isinstance(data.get("rationale"), str) or not data["rationale"].strip():
        raise ValueError("Rationale invalide")

    return TriageResult(**data)

example_output = {
    "category": "API",
    "urgency": "HIGH",
    "rationale": "Erreur 500 affectant une part importante des requêtes."
}

print(validate_result(example_output))

TriageResult(category='API', urgency='HIGH', rationale='Erreur 500 affectant une part importante des requêtes.')


### Explication du code
- Les ensembles `ALLOWED_*` définissent les valeurs autorisées.
- `TriageResult` représente une sortie structurée.
- `validate_result` rejette les catégories ou urgences inattendues.
- Une réponse textuelle vide est également refusée.

**Réflexe production :** le LLM propose ; le logiciel valide.

# 11. Notebook → API FastAPI

La cellule suivante **écrit** un petit fichier `app.py`. Elle ne démarre pas automatiquement un serveur dans le notebook.

In [17]:
app_code = r'''
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="ESI AI Inference API", version="1.0.0")

class PredictRequest(BaseModel):
    text: str

class PredictResponse(BaseModel):
    label: str
    score: float
    model_version: str

def predict(text: str):
    # À remplacer par votre vraie chaîne d'inférence.
    label = "LONG_TEXT" if len(text) > 80 else "SHORT_TEXT"
    score = 1.0
    return label, score

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict_endpoint(payload: PredictRequest):
    label, score = predict(payload.text)
    return {
        "label": label,
        "score": score,
        "model_version": "demo-1.0"
    }
'''
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print(app_code)


from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="ESI AI Inference API", version="1.0.0")

class PredictRequest(BaseModel):
    text: str

class PredictResponse(BaseModel):
    label: str
    score: float
    model_version: str

def predict(text: str):
    # À remplacer par votre vraie chaîne d'inférence.
    label = "LONG_TEXT" if len(text) > 80 else "SHORT_TEXT"
    score = 1.0
    return label, score

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictResponse)
def predict_endpoint(payload: PredictRequest):
    label, score = predict(payload.text)
    return {
        "label": label,
        "score": score,
        "model_version": "demo-1.0"
    }



### Explication du code
- `FastAPI` crée le service HTTP.
- `Pydantic` valide automatiquement le schéma d'entrée.
- `/health` permet à l'infrastructure de vérifier que le service répond.
- `/predict` expose l'inférence.
- `model_version` rend la réponse traçable.
- La fonction `predict()` est volontairement remplaçable par un vrai modèle.

Pour lancer localement après installation de FastAPI/Uvicorn :

`uvicorn app:app --reload`

# 12. Docker : rendre l'environnement reproductible

In [18]:
dockerfile = '''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''.strip()

requirements = '''
fastapi
uvicorn[standard]
pydantic
'''.strip()

with open("Dockerfile", "w", encoding="utf-8") as f:
    f.write(dockerfile)

with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

print("Dockerfile :\n")
print(dockerfile)
print("\nrequirements.txt :\n")
print(requirements)

Dockerfile :

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

requirements.txt :

fastapi
uvicorn[standard]
pydantic


### Explication du code
Le `Dockerfile` décrit comment construire l'image :

- image Python légère ;
- répertoire de travail `/app` ;
- installation des dépendances ;
- copie de l'application ;
- exposition du port 8000 ;
- démarrage d'Uvicorn.

Commandes typiques dans un terminal :

`docker build -t esi-ai-api .`

puis :

`docker run -p 8000:8000 esi-ai-api`

# 13. Monitoring : mesurer le service, pas seulement le modèle

In [19]:
import time
from collections import defaultdict

metrics = defaultdict(float)

def monitored_predict(text):
    start = time.perf_counter()

    try:
        label = "LONG_TEXT" if len(text) > 80 else "SHORT_TEXT"
        metrics["requests_total"] += 1
        metrics["success_total"] += 1
        return {"label": label}
    except Exception:
        metrics["requests_total"] += 1
        metrics["errors_total"] += 1
        raise
    finally:
        metrics["latency_seconds_total"] += time.perf_counter() - start

for sample in ["Bonjour", "A" * 120, "Test API"]:
    print(monitored_predict(sample))

print("\nMétriques :", dict(metrics))
print(
    "Latence moyenne :",
    metrics["latency_seconds_total"] / max(metrics["requests_total"], 1)
)

{'label': 'SHORT_TEXT'}
{'label': 'LONG_TEXT'}
{'label': 'SHORT_TEXT'}

Métriques : {'requests_total': 3.0, 'success_total': 3.0, 'latency_seconds_total': 1.3799995940644294e-05}
Latence moyenne : 4.599998646881431e-06


### Explication du code
Le wrapper mesure :

- le nombre total de requêtes ;
- le nombre de succès ;
- le nombre d'erreurs ;
- le temps cumulé d'inférence.

On peut ensuite calculer une latence moyenne.

**En production**, on ajouterait des métriques Prometheus/OpenTelemetry, des percentiles de latence, des alertes, le drift des données, la version du modèle et, pour un LLM, le nombre de tokens et le coût.

# 14. Checklist sécurité, RGPD et AI Act

Avant de mettre un système GenAI en production, demander :

### Sécurité
- Les entrées utilisateur sont-elles considérées comme non fiables ?
- Les permissions du modèle/outillage sont-elles minimales ?
- Les secrets sont-ils hors du code et des prompts ?
- Les sorties sont-elles validées avant action ?
- Existe-t-il un mécanisme de fallback ?

### Données / RGPD
- Avons-nous besoin de toutes les données collectées ?
- Quelle est la finalité ?
- Quelle est la durée de conservation ?
- Des données personnelles sont-elles envoyées à un fournisseur externe ?
- Les accès et traitements sont-ils traçables ?

### Gouvernance / AI Act
- Quel est le rôle exact du système dans la décision ?
- Le cas d'usage doit-il respecter des obligations de transparence ?
- Le système entre-t-il dans une catégorie réglementée ou à haut risque ?
- Les versions, tests, limites et incidents sont-ils documentés ?

> Pour un projet réel, la qualification réglementaire doit être faite à partir du cas d'usage concret et des textes applicables au moment du déploiement.

# 15. Mini-défi de fin de séance

À partir de ce notebook, construire un **assistant de triage de tickets** :

1. définir un schéma d'entrée ;
2. définir un schéma de sortie ;
3. écrire un prompt robuste ;
4. valider systématiquement la sortie ;
5. exposer `/predict` avec FastAPI ;
6. ajouter `/health` ;
7. préparer le Dockerfile ;
8. journaliser latence, erreurs et version ;
9. identifier au moins **3 risques** et leurs contrôles.

### Question finale
**Qu'est-ce qui fait qu'un prototype de modèle devient un système d'IA professionnel ?**

Réponse attendue : pas uniquement le score du modèle, mais l'ensemble **données + protocole + logiciel + sécurité + monitoring + gouvernance**.

---
## Références du Support 2 mobilisées

- Cornell University / Cornell Tech — **CS 5788: Introduction to Generative Models** : modèles autorégressifs, LLM, tokenisation, post-training et diffusion.
- Vaswani et al. — **Attention Is All You Need** (Transformer).
- Commission européenne — principes du **RGPD** et cadre d'application de l'**AI Act**.

Les schémas détaillés et crédits visuels figurent dans le Support 2. Ce notebook privilégie des implémentations pédagogiques originales afin de permettre l'explication ligne par ligne.